# 03 · Compare Python and SQL analysis

**Read-only SQL checks.** Run SQL 03–10 before this notebook. Input: core_analysis.pkl and populated SQL analytical views. Compare player IDs, counts, per-90 rates, percentiles, scores and ranks; a mismatch raises an error.

Run cells from top to bottom in a **fresh kernel**. The setup cell declares inputs explicitly. Code comments and explanations are in English. The original source cell numbers are recorded in notebook metadata and in `docs/cell_mapping.csv`.

## Setup

Locate the repository and import shared utilities. This does not change the database.

In [ ]:
from pathlib import Path
import sys
import json
from urllib.request import urlopen
import numpy as np
import pandas as pd
from IPython.display import display

# Find the repository from any notebook subfolder.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "scouting" / "config.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook inside the extracted repository.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from scouting.config import PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, REPORTS_DIR, IMPECT_BASE_URL, ITERATION_ID
from scouting.data_access import load_or_download_json, load_metadata_context, metadata
from scouting.checkpoints import save_context, load_context, BASE_VARIABLES, ADVANCED_VARIABLES
print("Project:", PROJECT_ROOT)

globals().update(load_context("core_analysis"))
import pyodbc
from scouting.database import connection_string as build_connection_string
connection_string = build_connection_string()


## Reconcile counts and rates

Read SQL views and compare candidate keys plus 13 numerical metrics against independently prepared Python results.

In [ ]:
import numpy as np

# SQL column → corresponding Python column.
metric_mapping = {
    "midfield_minutes": "midfield_minutes",

    "forward_pass_10m_count": "successful_forward_10m_count",
    "forward_carry_10m_count": "successful_forward_carry_10m_count",
    "final_third_pass_entry_count": "final_third_pass_entries",
    "final_third_carry_entry_count": "final_third_carry_entries",
    "interception_count": "interceptions",
    "loose_ball_regain_count": "loose_ball_regains",

    "forward_pass_10m_per90": "successful_forward_10m_per90",
    "forward_carry_10m_per90": "successful_forward_carry_10m_per90",
    "final_third_pass_entries_per90": "final_third_pass_entries_per90",
    "final_third_carry_entries_per90": "final_third_carry_entries_per90",
    "interceptions_per90": "interceptions_per90",
    "loose_ball_regains_per90": "loose_ball_regains_per90",
}

# Read unrounded SQL values.
connection = pyodbc.connect(connection_string, timeout=10)

try:
    cursor = connection.cursor()

    query = (
        "SELECT player_id, "
        + ", ".join(metric_mapping)
        + " FROM analytics.vw_midfield_metrics "
        + "WHERE iteration_id = ?;"
    )

    cursor.execute(query, 743)

    sql_metrics = pd.DataFrame.from_records(
        [tuple(row) for row in cursor.fetchall()],
        columns=[column[0] for column in cursor.description],
    )

finally:
    connection.close()

# Give both tables consistent column names before comparison.
python_metrics = midfield_profile[
    ["player.id", "player_name"] + list(metric_mapping.values())
].rename(columns={"player.id": "player_id"}).copy()

sql_metrics = sql_metrics.rename(columns={
    sql_column: f"{python_column}_sql"
    for sql_column, python_column in metric_mapping.items()
})

comparison = python_metrics.merge(
    sql_metrics,
    on="player_id",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

print("Python candidates:", len(python_metrics))
print("SQL candidates:", len(sql_metrics))

unmatched = comparison.loc[comparison["_merge"].ne("both")]

if not unmatched.empty:
    display(unmatched[["player_id", "player_name", "_merge"]])
    raise ValueError("SQL and Python candidate IDs differ.")

print("Candidate IDs match: YES")

# Exact agreement for counts; small tolerance for decimal arithmetic.
check_rows = []
mismatch_details = []

for sql_column, python_column in metric_mapping.items():
    python_values = pd.to_numeric(comparison[python_column])
    sql_values = pd.to_numeric(
        comparison[f"{python_column}_sql"]
    )

    if sql_column.endswith("_count"):
        tolerance = 0
    elif sql_column == "midfield_minutes":
        tolerance = 0.00001
    else:
        tolerance = 0.000001

    matches = np.isclose(
        python_values.to_numpy(dtype=float),
        sql_values.to_numpy(dtype=float),
        atol=tolerance,
        rtol=0,
        equal_nan=False,
    )

    differences = (python_values - sql_values).abs()

    check_rows.append({
        "metric": sql_column,
        "players_checked": len(comparison),
        "mismatches": int((~matches).sum()),
        "max_absolute_difference": differences.max(),
    })

    if not matches.all():
        detail = comparison.loc[
            ~matches, ["player_id", "player_name"]
        ].copy()

        detail["metric"] = sql_column
        detail["python_value"] = python_values.loc[~matches]
        detail["sql_value"] = sql_values.loc[~matches]

        mismatch_details.append(detail)

reconciliation_report = pd.DataFrame(check_rows)

display(reconciliation_report)

if mismatch_details:
    display(pd.concat(mismatch_details, ignore_index=True))
else:
    print("All 13 measures reconcile for all candidates.")
if mismatch_details:
    raise AssertionError("Counts/rates do not reconcile; inspect mismatch details.")


## Inspect the passing-rank tie

Examine Schlager and Xhaka to illustrate why tiny floating-point differences should not split a mathematical tie.

In [ ]:
connection = pyodbc.connect(connection_string, timeout=10)

try:
    cursor = connection.cursor()

    cursor.execute("""
        SELECT
            player_id,
            passing_focus_score,
            passing_focus_rank
        FROM analytics.vw_midfield_screening
        WHERE iteration_id = 743
          AND player_id IN (7873, 281);
    """)

    sql_passing_scores = pd.DataFrame.from_records(
        [tuple(row) for row in cursor.fetchall()],
        columns=[column[0] for column in cursor.description],
    )

finally:
    connection.close()

python_passing_scores = midfield_screening.loc[
    midfield_screening["player.id"].isin([7873, 281]),
    [
        "player.id",
        "player_name",
        "passing_focus_score",
        "passing_focus_rank",
    ],
].rename(columns={"player.id": "player_id"})

score_check = python_passing_scores.merge(
    sql_passing_scores,
    on="player_id",
    suffixes=("_python", "_sql"),
    validate="one_to_one",
)

score_check["passing_focus_score_sql"] = pd.to_numeric(
    score_check["passing_focus_score_sql"]
)

print(
    score_check.to_string(
        index=False,
        float_format=lambda value: f"{value:.12f}",
    )
)

## Reconcile percentiles, scores and ranks

Use the same six-decimal tie rule in both systems and compare 12 analytical outputs for all candidates.

In [ ]:
# Use the same six-decimal tie rule in the Python comparison.
python_screening_check = midfield_screening.copy()

for scenario in ["balanced", "passing_focus", "carrying_focus"]:
    python_screening_check[f"{scenario}_rank"] = (
        python_screening_check[f"{scenario}_score"]
        .round(6)
        .rank(ascending=False, method="min")
        .astype(int)
    )

check_columns = [
    "forward_pass_pctile",
    "forward_carry_pctile",
    "third_entry_pass_pctile",
    "third_entry_carry_pctile",
    "interception_pctile",
    "loose_regain_pctile",
    "balanced_score",
    "passing_focus_score",
    "carrying_focus_score",
    "balanced_rank",
    "passing_focus_rank",
    "carrying_focus_rank",
]

# Read SQL results without display rounding.
connection = pyodbc.connect(connection_string, timeout=10)

try:
    cursor = connection.cursor()

    query = (
        "SELECT player_id, "
        + ", ".join(check_columns)
        + " FROM analytics.vw_midfield_screening "
        + "WHERE iteration_id = ?;"
    )

    cursor.execute(query, 743)

    sql_screening_check = pd.DataFrame.from_records(
        [tuple(row) for row in cursor.fetchall()],
        columns=[column[0] for column in cursor.description],
    )

finally:
    connection.close()

python_table = python_screening_check[
    ["player.id", "player_name"] + check_columns
].rename(columns={"player.id": "player_id"})

screening_comparison = python_table.merge(
    sql_screening_check,
    on="player_id",
    how="outer",
    suffixes=("_python", "_sql"),
    validate="one_to_one",
    indicator=True,
)

print("Python candidates:", len(python_table))
print("SQL candidates:", len(sql_screening_check))

if not screening_comparison["_merge"].eq("both").all():
    display(
        screening_comparison.loc[
            screening_comparison["_merge"].ne("both"),
            ["player_id", "player_name", "_merge"],
        ]
    )
    raise ValueError("Candidate IDs do not match.")

check_results = []
mismatch_tables = []

for column in check_columns:
    python_values = pd.to_numeric(
        screening_comparison[f"{column}_python"]
    ).astype(float)

    sql_values = pd.to_numeric(
        screening_comparison[f"{column}_sql"]
    ).astype(float)

    # Ranks must agree exactly.
    tolerance = 0 if column.endswith("_rank") else 0.000001

    differences = (python_values - sql_values).abs()

    matches = (
        python_values.notna()
        & sql_values.notna()
        & differences.le(tolerance)
    )

    check_results.append({
        "measure": column,
        "players_checked": len(screening_comparison),
        "mismatches": int((~matches).sum()),
        "max_absolute_difference": differences.max(),
    })

    if not matches.all():
        details = screening_comparison.loc[
            ~matches, ["player_id", "player_name"]
        ].copy()

        details["measure"] = column
        details["python_value"] = python_values.loc[~matches]
        details["sql_value"] = sql_values.loc[~matches]

        mismatch_tables.append(details)

screening_reconciliation = pd.DataFrame(check_results)
display(screening_reconciliation)

if mismatch_tables:
    display(pd.concat(mismatch_tables, ignore_index=True))
else:
    print("All percentiles, scores, and ranks reconcile.")
if mismatch_tables:
    raise AssertionError("Screening outputs do not reconcile; inspect mismatch details.")


## Export reconciliation evidence

Save named intermediate tables for the next notebook. Generated files stay local and are excluded from Git.

In [ ]:
reconciliation_report.to_csv(REPORTS_DIR / "python_sql_metric_validation.csv", index=False)
screening_reconciliation.to_csv(REPORTS_DIR / "python_sql_screening_validation.csv", index=False)
